# Phase 1: Data Acquisition & Corpus Construction

This notebook walks through **Phase 1** of the pipeline:
1. Scraping curated **TikTok Viral (`label=1`)** and **Non-Viral Commercial (`label=0`)** Spotify playlists (`config/playlists_viral.txt` and `config/playlists_non_viral.txt`).
2. Filtering tracks strictly to release years **2020–2025** and deduplicating on normalized `(primary_artist, clean_title)`.
3. Matching tracks strictly against Hugging Face's `theelderemo/genius-lyrics-cleaned` corpus.
4. Creating the balanced dataset (up to 3,000 Viral / 3,000 Non-Viral) and stratified 80/20 train/test split.

In [ ]:
import sys
from pathlib import Path

# Add repository root to sys.path
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data.normalize import make_match_key, normalize_artist, normalize_title
from src.data.spotify_scraper import run_spotify_scraper
from src.data.genius_matcher import match_spotify_to_genius
from src.data.build_dataset import print_collection_report, build_balanced_splits

## 1. Sanity-Check Entity Normalization
Verify how `normalize_artist`, `normalize_title`, and `make_match_key` collapse remixes, sped-up versions, and `(feat. ...)` tags for strict exact matching.

In [ ]:
examples = [
    ("Doja Cat", "Say So (feat. Nicki Minaj)"),
    ("FIFTY FIFTY", "Cupid - Twin Ver. (Sped Up)"),
    ("Lil Nas X & Jack Harlow", "INDUSTRY BABY (with Jack Harlow)"),
    ("The Weeknd", "Blinding Lights - 2020 Remix"),
]

pd.DataFrame([
    {
        "raw_artist": a,
        "raw_title": t,
        "norm_artist": normalize_artist(a),
        "norm_title": normalize_title(t),
        "match_key": make_match_key(a, t),
    }
    for a, t in examples
])

## 2. Scrape Curated Spotify Playlists (2020–2025)
Make sure `.env` contains `SPOTIPY_CLIENT_ID` and `SPOTIPY_CLIENT_SECRET`, and add playlist URLs/IDs to:
- `config/playlists_viral.txt`
- `config/playlists_non_viral.txt`

In [ ]:
viral_sp, non_viral_sp = run_spotify_scraper(min_year=2020, max_year=2025, target_per_class=3000)

## 3. Match Against `theelderemo/genius-lyrics-cleaned` & Build 80/20 Splits

In [ ]:
combined_sp = pd.concat([viral_sp, non_viral_sp], ignore_index=True)
if not combined_sp.empty:
    matched_all, unmatched_all = match_spotify_to_genius(combined_sp, streaming=True)
    viral_matched = matched_all[matched_all["label"] == 1].copy()
    non_viral_matched = matched_all[matched_all["label"] == 0].copy()

    print_collection_report(viral_sp, non_viral_sp, viral_matched, non_viral_matched, target_per_class=3000)
    full_df, train_df, test_df = build_balanced_splits(viral_matched, non_viral_matched, target_per_class=3000)